# YOLO Segmentation — Dataset Exploration, Repair, Augmentation, dan Clean Export

Notebook ini dibuat untuk dataset `nutrivision-pro-yolo-v7` di Kaggle.

Alur:

1. Temukan dataset dan baca `data.yaml`
2. Hitung overview, distribusi kelas, dimensi gambar, dan pairing image-label
3. Validasi format YOLO segmentation secara aman
4. Analisis instance dan geometry polygon
5. Tampilkan kandidat visual QA
6. Buat template keputusan manual untuk kasus ambigu
7. Lakukan repair konservatif tanpa mengubah dataset asli
8. **[BARU] Class-Aware Copy-Paste Augmentation** untuk kelas minoritas
9. **[BARU] Targeted Oversampling Manifest** — gandakan entri gambar kelas langka di train.txt
10. Re-audit dan ekspor `clean_v1/`

**Catatan penting:** format detection `class x_center y_center width height` tidak
dikonversi menjadi polygon secara otomatis — baris tersebut dicatat dan dibuang
dari export segmentation.

**Kelas target augmentasi** (berdasarkan hasil evaluasi, miss rate tinggi):
`sambal`, `tofu`, `squid`, `fish`, `tempeh`, `chicken`, `egg`, `fruit`


## 0. Setup dan konfigurasi

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image, ImageDraw
from IPython.display import display

INPUT_ROOT  = Path("/kaggle/input")
WORK_ROOT   = Path("/kaggle/working")
DATASET_HINT = "nutrivision-pro-yolo-v7"

CLEAN_DIR              = WORK_ROOT / "clean_v1"
AUDIT_DIR              = WORK_ROOT / "yolo_audit"
REVIEW_DECISIONS_PATH  = None

AUTO_REMOVE_INVALID             = True
AUTO_REMOVE_EXACT_DUPLICATES    = True
AUTO_REMOVE_DETECTION_LINES     = True
CREATE_EMPTY_LABEL_FOR_UNPAIRED_IMAGE = True
OVERWRITE_OUTPUT                = True

TINY_POLYGON_AREA       = 0.00005
TINY_BBOX_AREA          = 0.00010
LOW_MASK_BBOX_RATIO     = 0.05
COMPLEX_POLYGON_POINTS  = 150
NEAR_DUPLICATE_BBOX_IOU = 0.98

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_NAMES      = ("train", "valid", "test")
RANDOM_SEED      = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Augmentation / oversampling config ──────────────────────────────────────
# Kelas yang mendapat perlakuan khusus (diurutkan dari paling kritis)
MINORITY_CLASSES = [
    "sambal", "tofu", "squid", "fish",
    "tempeh", "chicken", "egg", "fruit",
]

# Jumlah gambar sintetis yang dibuat per kelas minority via copy-paste
COPY_PASTE_PER_CLASS = 40

# Jumlah titik minimum pada donor polygon agar dianggap layak dipaste
DONOR_MIN_POINTS = 6

# Faktor repetisi di manifest oversampling (gambar minority muncul N kali)
OVERSAMPLE_FACTOR = 4

# ── Seed global ─────────────────────────────────────────────────────────────
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration ready.")


## 1. Dataset setup — locate dataset, load YAML, dan class names

In [ ]:
def find_dataset_yaml(input_root=INPUT_ROOT, hint=DATASET_HINT):
    candidates = sorted(input_root.rglob("data.yaml"))
    if not candidates:
        raise FileNotFoundError(f"Tidak menemukan data.yaml di {input_root}")

    def score(path):
        text = str(path).lower()
        sibling_score = sum(
            int((path.parent / split / "images").is_dir())
            for split in ("train", "valid", "val", "test")
        )
        return (int(hint.lower() in text) * 100 + sibling_score * 10, -len(text))

    return max(candidates, key=score)


DATA_YAML    = find_dataset_yaml()
DATASET_DIR  = DATA_YAML.parent
with DATA_YAML.open("r", encoding="utf-8") as f:
    DATA_CONFIG = yaml.safe_load(f) or {}


def class_names_from_yaml(config):
    names = config.get("names", [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=lambda x: int(x))]
    names = [str(name) for name in names]
    if not names:
        raise ValueError("data.yaml tidak memiliki names.")
    return names


CLASS_NAMES = class_names_from_yaml(DATA_CONFIG)
NUM_CLASSES = int(DATA_CONFIG.get("nc", len(CLASS_NAMES)))
if NUM_CLASSES != len(CLASS_NAMES):
    print(f"Peringatan: nc={NUM_CLASSES}, tetapi names={len(CLASS_NAMES)}.")
    NUM_CLASSES = len(CLASS_NAMES)

CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def resolve_split_dirs(dataset_root, config, output_name, config_keys, aliases):
    raw_value = next((config.get(key) for key in config_keys if config.get(key)), None)
    candidates = []
    if raw_value:
        values = raw_value if isinstance(raw_value, list) else [raw_value]
        for value in values:
            path = Path(str(value))
            if not path.is_absolute():
                path = dataset_root / path
            candidates.append(path)
    for alias in aliases:
        candidates.extend([
            dataset_root / alias / "images",
            dataset_root / alias,
        ])
    for candidate in candidates:
        image_dir = candidate if candidate.name == "images" else candidate / "images"
        label_dir = image_dir.parent / "labels"
        if image_dir.is_dir() and label_dir.is_dir():
            return {"name": output_name, "root": image_dir.parent,
                    "images": image_dir, "labels": label_dir}
    raise FileNotFoundError(f"Split {output_name} tidak ditemukan.")


SOURCE_SPLITS = {
    "train": resolve_split_dirs(DATASET_DIR, DATA_CONFIG, "train", ("train",), ("train",)),
    "valid": resolve_split_dirs(DATASET_DIR, DATA_CONFIG, "valid", ("val", "valid"), ("valid", "val")),
    "test":  resolve_split_dirs(DATASET_DIR, DATA_CONFIG, "test",  ("test",),  ("test",)),
}

print("Dataset YAML :", DATA_YAML)
print("Dataset root :", DATASET_DIR)
print("Classes      :", NUM_CLASSES)
for name, split in SOURCE_SPLITS.items():
    print(f"{name:5} images={split['images']} labels={split['labels']}")
print("\nClass names:")
for class_id, name in enumerate(CLASS_NAMES):
    print(f"  {class_id}: {name}")


## 2. Dataset overview — image count, label count, dimensions, dan distribusi kelas

In [ ]:
def image_files(image_dir):
    return sorted(
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

def label_files(label_dir):
    return sorted(p for p in label_dir.rglob("*.txt") if p.is_file())

def image_index(image_dir):
    index = defaultdict(list)
    for path in image_files(image_dir):
        index[path.stem].append(path)
    return index

def inspect_images(split_name, split):
    records = []
    for path in image_files(split["images"]):
        row = {"split": split_name, "file": path.name, "stem": path.stem,
               "path": str(path), "width": None, "height": None,
               "aspect_ratio": None, "readable": False, "error": ""}
        try:
            with Image.open(path) as image:
                image.load()
                width, height = image.size
            row.update(width=width, height=height, aspect_ratio=width / height, readable=True)
        except Exception as exc:
            row["error"] = repr(exc)
        records.append(row)
    return records

image_info = pd.DataFrame(
    [row for split_name, split in SOURCE_SPLITS.items()
     for row in inspect_images(split_name, split)]
)

overview_rows = []
for split_name, split in SOURCE_SPLITS.items():
    images = image_files(split["images"])
    labels = label_files(split["labels"])
    overview_rows.append({
        "split": split_name,
        "images": len(images),
        "label_files": len(labels),
        "readable_images": int(image_info.loc[image_info["split"].eq(split_name), "readable"].sum()),
        "empty_label_files": sum(p.stat().st_size == 0 for p in labels),
        "image_stems": len({p.stem for p in images}),
        "label_stems": len({p.stem for p in labels}),
    })

overview_df = pd.DataFrame(overview_rows)
display(overview_df)
display(image_info[["width", "height", "aspect_ratio"]].describe(include="all"))

if not image_info.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(data=image_info.dropna(subset=["width"]), x="width", hue="split", bins=30, ax=axes[0])
    sns.histplot(data=image_info.dropna(subset=["height"]), x="height", hue="split", bins=30, ax=axes[1])
    axes[0].set_title("Image width")
    axes[1].set_title("Image height")
    plt.tight_layout()
    plt.show()


## 3. Annotation validation

In [ ]:
def canonical_number(value):
    value = float(value)
    if value == 0:
        value = 0.0
    return f"{value:.10f}".rstrip("0").rstrip(".") or "0"

def canonical_line(class_id, coords):
    return " ".join([str(int(class_id))] + [canonical_number(x) for x in coords])

def parse_annotation_line(raw_line, line_number, class_names):
    text = raw_line.strip()
    result = {
        "line_number": line_number, "raw": text,
        "format": "blank" if not text else "unknown",
        "valid": False, "reasons": [],
        "class_id": None, "coords": None, "num_points": None, "canonical": None,
    }
    if not text:
        return result
    tokens = text.split()
    if len(tokens) == 5:
        result["format"] = "detection"
        result["reasons"] = ["detection_format_not_segmentation"]
        return result
    if len(tokens) < 7:
        result["reasons"] = ["too_few_values_for_segmentation"]
        return result
    result["format"] = "segmentation"
    try:
        class_id = int(tokens[0])
    except Exception:
        result["reasons"] = ["class_id_not_integer"]
        return result
    result["class_id"] = class_id
    if not 0 <= class_id < len(class_names):
        result["reasons"].append("class_id_out_of_range")
    try:
        coords = [float(x) for x in tokens[1:]]
    except Exception:
        result["reasons"].append("coordinate_not_numeric")
        return result
    if not all(math.isfinite(x) for x in coords):
        result["reasons"].append("coordinate_not_finite")
    if len(coords) % 2 != 0:
        result["reasons"].append("odd_coordinate_count")
    result["coords"] = coords
    result["num_points"] = len(coords) // 2
    if result["num_points"] < 3:
        result["reasons"].append("fewer_than_3_polygon_points")
    if coords and not all(0.0 <= x <= 1.0 for x in coords):
        result["reasons"].append("coordinate_out_of_range_0_1")
    if len(coords) % 2 == 0 and len(coords) >= 6:
        result["canonical"] = canonical_line(class_id, coords)
    result["valid"] = len(result["reasons"]) == 0
    if result["valid"] and result["canonical"] is None:
        result["valid"] = False
    return result

def collect_validation(split_dirs, class_names):
    rows = []
    for split_name, split in split_dirs.items():
        for label_path in label_files(split["labels"]):
            with label_path.open("r", encoding="utf-8", errors="replace") as f:
                physical_lines = f.readlines()
            if not physical_lines:
                rows.append({
                    "split": split_name, "label_file": label_path.name,
                    "label_path": str(label_path), "line_number": 0,
                    "raw": "", "format": "empty_file", "valid": False,
                    "reasons": ["empty_label_file"], "class_id": None,
                    "coords": None, "num_points": None, "canonical": None,
                    "exact_duplicate": False,
                })
                continue
            seen = set()
            for line_number, raw in enumerate(physical_lines, start=1):
                parsed = parse_annotation_line(raw, line_number, class_names)
                if parsed["format"] == "blank":
                    continue
                parsed.update({"split": split_name, "label_file": label_path.name,
                                "label_path": str(label_path)})
                duplicate = bool(parsed["valid"] and parsed["canonical"] in seen)
                if parsed["valid"]:
                    seen.add(parsed["canonical"])
                parsed["exact_duplicate"] = duplicate
                rows.append(parsed)
    return pd.DataFrame(rows)

validation_df = collect_validation(SOURCE_SPLITS, CLASS_NAMES)
validation_df["reason_text"] = validation_df["reasons"].apply(lambda x: ";".join(x))

def pairing_table(split_dirs):
    rows = []
    for split_name, split in split_dirs.items():
        image_stems = {p.stem for p in image_files(split["images"])}
        label_stems = {p.stem for p in label_files(split["labels"])}
        rows.append({
            "split": split_name,
            "images_without_labels": len(image_stems - label_stems),
            "labels_without_images": len(label_stems - image_stems),
            "paired_stems": len(image_stems & label_stems),
            "unpaired_image_names": sorted(image_stems - label_stems),
            "orphan_label_names": sorted(label_stems - image_stems),
        })
    return pd.DataFrame(rows)

pairing_df = pairing_table(SOURCE_SPLITS)
format_df = (
    validation_df.groupby(["split", "format"], dropna=False).size()
    .reset_index(name="lines")
    if not validation_df.empty else pd.DataFrame()
)
error_df = (
    validation_df[~validation_df["valid"]]
    .explode("reasons")
    .groupby(["split", "reasons"]).size()
    .reset_index(name="count")
    if not validation_df.empty else pd.DataFrame()
)
print("Format counts:")
display(format_df)
print("Pairing:")
display(pairing_df.drop(columns=["unpaired_image_names", "orphan_label_names"]))
print("Validation errors:")
display(error_df)


## 4. Instance analysis — distribusi kelas dan multi-instance

In [ ]:
valid_annotations = validation_df[validation_df["valid"]].copy()

if valid_annotations.empty:
    instance_df = pd.DataFrame(columns=["split","label_file","class_id","class_name","instance_count"])
else:
    instance_df = (
        valid_annotations.groupby(["split","label_file","class_id"])
        .size().reset_index(name="instance_count")
    )
    instance_df["class_name"] = instance_df["class_id"].map(dict(enumerate(CLASS_NAMES)))

class_distribution = (
    valid_annotations.groupby(["split","class_id"]).size().reset_index(name="instances")
    if not valid_annotations.empty
    else pd.DataFrame(columns=["split","class_id","instances"])
)
if not class_distribution.empty:
    class_distribution["class_name"] = class_distribution["class_id"].map(dict(enumerate(CLASS_NAMES)))
    class_distribution = class_distribution.sort_values(["split","class_id"])

multi_instance = instance_df[instance_df["instance_count"] > 1].copy()
print("Class distribution:")
display(class_distribution)
print("Image/class pairs dengan multiple instances:")
display(multi_instance.sort_values("instance_count", ascending=False).head(50))

if not class_distribution.empty:
    plt.figure(figsize=(14, 5))
    sns.barplot(data=class_distribution, x="class_name", y="instances", hue="split")
    plt.xticks(rotation=45, ha="right")
    plt.title("Valid segmentation instances by class")
    plt.tight_layout()
    plt.show()


## 5. Segmentation geometry

In [ ]:
def shoelace_area(coords):
    points = np.asarray(coords, dtype=float).reshape(-1, 2)
    x, y = points[:, 0], points[:, 1]
    return float(0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))

def geometry_from_record(row):
    coords = row["coords"]
    points = np.asarray(coords, dtype=float).reshape(-1, 2)
    x_min, y_min = points.min(axis=0)
    x_max, y_max = points.max(axis=0)
    bbox_width   = float(x_max - x_min)
    bbox_height  = float(y_max - y_min)
    polygon_area = shoelace_area(coords)
    bbox_area    = bbox_width * bbox_height
    ratio        = polygon_area / bbox_area if bbox_area > 0 else 0.0
    return {
        "split": row["split"], "label_file": row["label_file"],
        "line_number": row["line_number"], "class_id": row["class_id"],
        "class_name": CLASS_NAMES[int(row["class_id"])],
        "num_points": int(row["num_points"]),
        "polygon_area": polygon_area, "bbox_width": bbox_width,
        "bbox_height": bbox_height, "bbox_area": bbox_area,
        "mask_bbox_ratio": ratio,
        "tiny_polygon": polygon_area < TINY_POLYGON_AREA,
        "tiny_bbox": bbox_area < TINY_BBOX_AREA,
        "low_mask_bbox_ratio": ratio < LOW_MASK_BBOX_RATIO,
        "complex_polygon": int(row["num_points"]) >= COMPLEX_POLYGON_POINTS,
        "canonical": row["canonical"],
    }

geometry_df = pd.DataFrame([geometry_from_record(row) for _, row in valid_annotations.iterrows()])
if not geometry_df.empty:
    geometry_df["multi_instance_same_class"] = geometry_df.set_index(
        ["split","label_file","class_id"]
    ).index.isin(
        multi_instance.set_index(["split","label_file","class_id"]).index
    )
    geometry_df["review_reason"] = geometry_df.apply(
        lambda r: ";".join(name for name, enabled in (
            ("tiny_polygon", r["tiny_polygon"]),
            ("tiny_bbox", r["tiny_bbox"]),
            ("low_mask_bbox_ratio", r["low_mask_bbox_ratio"]),
            ("complex_polygon", r["complex_polygon"]),
            ("multiple_same_class", r["multi_instance_same_class"]),
        ) if enabled), axis=1,
    )

class_geometry_stats = (
    geometry_df.groupby(["class_id","class_name"]).agg(
        instances=("class_id","size"),
        polygon_area_median=("polygon_area","median"),
        bbox_area_median=("bbox_area","median"),
        mask_bbox_ratio_median=("mask_bbox_ratio","median"),
        points_median=("num_points","median"),
        tiny_polygon=("tiny_polygon","sum"),
        tiny_bbox=("tiny_bbox","sum"),
        low_ratio=("low_mask_bbox_ratio","sum"),
    ).reset_index()
    if not geometry_df.empty else pd.DataFrame()
)
display(class_geometry_stats)

if not geometry_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.histplot(geometry_df["polygon_area"], bins=40, ax=axes[0])
    sns.histplot(geometry_df["bbox_area"], bins=40, ax=axes[1])
    sns.histplot(geometry_df["mask_bbox_ratio"], bins=40, ax=axes[2])
    axes[0].set_title("Polygon area")
    axes[1].set_title("BBox area")
    axes[2].set_title("Mask / bbox ratio")
    plt.tight_layout()
    plt.show()


## 6. Visual segmentation QA

In [ ]:
def find_image(split_name, stem):
    paths = image_index(SOURCE_SPLITS[split_name]["images"]).get(stem, [])
    return paths[0] if paths else None

def annotations_for_image(split_name, label_file):
    return valid_annotations[
        (valid_annotations["split"] == split_name)
        & (valid_annotations["label_file"] == label_file)
    ]

def show_annotation_group(records, title="", cols=3):
    records = list(records)
    if not records:
        print("Tidak ada kandidat untuk ditampilkan.")
        return
    n = len(records)
    rows_count = math.ceil(n / cols)
    fig, axes = plt.subplots(rows_count, cols, figsize=(cols * 5, rows_count * 5))
    axes = np.atleast_1d(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, record in zip(axes, records):
        image_path = find_image(record["split"], Path(record["label_file"]).stem)
        if image_path is None:
            ax.set_title("image not found")
            continue
        with Image.open(image_path).convert("RGB") as image:
            ax.imshow(image)
            width, height = image.size
            for _, ann in annotations_for_image(record["split"], record["label_file"]).iterrows():
                points = np.asarray(ann["coords"], dtype=float).reshape(-1, 2)
                px = points * np.array([width, height])
                color = "red" if ann["line_number"] == record["line_number"] else "yellow"
                closed = np.vstack([px, px[0]])
                ax.plot(closed[:, 0], closed[:, 1], color=color, linewidth=2)
                ax.text(px[0, 0], px[0, 1], CLASS_NAMES[int(ann["class_id"])],
                        color=color, fontsize=8, bbox={"facecolor": "black", "alpha": 0.5})
            ax.set_title(
                f"{record['split']}/{record['label_file']}:{record['line_number']}\n"
                f"{record.get('review_reason', record.get('reason_text', ''))}"
            )
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

random_candidates = valid_annotations.sample(
    min(12, len(valid_annotations)), random_state=RANDOM_SEED
).to_dict("records") if not valid_annotations.empty else []
show_annotation_group(random_candidates, "Random segmentation samples")

if not geometry_df.empty:
    suspicious = geometry_df[geometry_df["review_reason"].ne("")].copy()
    show_annotation_group(
        suspicious.sort_values(["tiny_polygon","low_mask_bbox_ratio"], ascending=False)
        .head(12).to_dict("records"),
        "Geometry review candidates",
    )


## 7. Repair plan dan keputusan manual

### Otomatis dan aman
- baris kosong diabaikan
- baris malformed, class ID invalid, koordinat non-numeric, jumlah titik ganjil, titik < 3, koordinat di luar `[0,1]` dibuang
- baris detection 5-token tidak dipakai
- exact duplicate dalam file yang sama dibuang
- image yang tidak bisa dibaca tidak diekspor

### Tidak otomatis
- polygon kecil, kompleks, rasio rendah tidak dibuang otomatis
- merge instance tidak dilakukan
- class relabel tidak ditebak

Buat `repair_decisions_template.csv`, isi `action` (keep / remove / replace), lalu set `REVIEW_DECISIONS_PATH` dan jalankan ulang.


In [ ]:
review_candidates = []
for _, row in validation_df.iterrows():
    if row["valid"] and row["exact_duplicate"]:
        continue
    if not row["valid"]:
        continue
    geo_row = geometry_df[
        (geometry_df["split"] == row["split"])
        & (geometry_df["label_file"] == row["label_file"])
        & (geometry_df["line_number"] == row["line_number"])
    ]
    if geo_row.empty:
        continue
    reason = geo_row.iloc[0]["review_reason"]
    if reason:
        review_candidates.append({
            "split": row["split"], "label_file": row["label_file"],
            "line_number": int(row["line_number"]),
            "action": "", "replacement_line": "",
            "reason": reason, "current_line": row["raw"],
        })

review_candidates_df = pd.DataFrame(review_candidates)
template_path = AUDIT_DIR / "repair_decisions_template.csv"
review_candidates_df.to_csv(template_path, index=False)
print(f"Template keputusan manual: {template_path}")
display(review_candidates_df.head(30))


## 8. Dataset repair — export ke clean_v1 tanpa mengubah source

In [ ]:
def parse_replacement_line(line):
    if not isinstance(line, str) or not line.strip():
        raise ValueError("replacement_line kosong.")
    parsed = parse_annotation_line(line, 0, CLASS_NAMES)
    if not parsed["valid"]:
        raise ValueError(f"replacement_line invalid: {parsed['reasons']}")
    return parsed

def load_manual_decisions(path_value):
    if path_value is None:
        return {}
    path = Path(path_value)
    if not path.exists():
        raise FileNotFoundError(f"File keputusan manual tidak ditemukan: {path}")
    decisions = pd.read_csv(path).fillna("")
    required = {"split","label_file","line_number","action","replacement_line"}
    missing = required - set(decisions.columns)
    if missing:
        raise ValueError(f"Kolom keputusan manual kurang: {sorted(missing)}")
    result = {}
    for _, row in decisions.iterrows():
        key = (str(row["split"]), str(row["label_file"]), int(row["line_number"]))
        action = str(row["action"]).strip().lower()
        if action not in {"keep","remove","replace"}:
            raise ValueError(f"Action harus keep/remove/replace, tetapi mendapat {action!r} untuk {key}")
        result[key] = {"action": action, "replacement_line": str(row["replacement_line"]).strip()}
    return result

MANUAL_DECISIONS = load_manual_decisions(REVIEW_DECISIONS_PATH)
print("Manual decisions loaded:", len(MANUAL_DECISIONS))

def normalise_output_line(parsed):
    return parsed["canonical"] + "\n"

def copy_clean_dataset(source_splits, output_dir, validation, manual_decisions):
    if output_dir.exists() and OVERWRITE_OUTPUT:
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    repair_log = []

    for split_name, split in source_splits.items():
        out_images = output_dir / split_name / "images"
        out_labels = output_dir / split_name / "labels"
        out_images.mkdir(parents=True, exist_ok=True)
        out_labels.mkdir(parents=True, exist_ok=True)

        img_by_stem   = image_index(split["images"])
        label_by_stem = {p.stem: p for p in label_files(split["labels"])}
        split_validation = validation[validation["split"] == split_name]

        for stem, candidates in sorted(img_by_stem.items()):
            source_image = candidates[0]
            out_image    = out_images / source_image.name
            readable = bool(
                image_info[(image_info["split"] == split_name) & (image_info["stem"] == stem)]["readable"].any()
            )
            if not readable:
                repair_log.append({"split": split_name, "file": source_image.name,
                                    "line_number": "", "action": "drop_unreadable_image",
                                    "reason": "image_cannot_be_read"})
                continue
            shutil.copy2(source_image, out_image)

            label_path = label_by_stem.get(stem)
            if label_path is None:
                (out_labels / f"{stem}.txt").write_text("", encoding="utf-8")
                repair_log.append({"split": split_name, "file": f"{stem}.txt",
                                    "line_number": "", "action": "create_empty_label",
                                    "reason": "image_without_label"})
                continue

            file_rows  = split_validation[split_validation["label_file"] == label_path.name]
            row_by_line = {int(row["line_number"]): row for _, row in file_rows.iterrows()}
            seen = set()
            output_lines = []
            with label_path.open("r", encoding="utf-8", errors="replace") as f:
                source_lines = f.readlines()

            for line_number, raw in enumerate(source_lines, start=1):
                if not raw.strip():
                    continue
                row = row_by_line.get(line_number)
                key = (split_name, label_path.name, line_number)
                decision = manual_decisions.get(key)
                parsed = parse_annotation_line(raw, line_number, CLASS_NAMES)

                if decision:
                    if decision["action"] == "remove":
                        repair_log.append({"split": split_name, "file": label_path.name,
                                            "line_number": line_number,
                                            "action": "manual_remove", "reason": "manual_decision"})
                        continue
                    if decision["action"] == "replace":
                        parsed = parse_replacement_line(decision["replacement_line"])
                        repair_log.append({"split": split_name, "file": label_path.name,
                                            "line_number": line_number,
                                            "action": "manual_replace", "reason": "manual_decision"})
                    elif not parsed["valid"]:
                        raise ValueError(f"Decision keep tidak boleh untuk baris invalid: {key}")

                if not parsed["valid"]:
                    if parsed["format"] == "detection" and not AUTO_REMOVE_DETECTION_LINES:
                        raise RuntimeError(f"Detection line, AUTO_REMOVE_DETECTION_LINES=False: {key}")
                    if parsed["format"] != "detection" and not AUTO_REMOVE_INVALID:
                        raise RuntimeError(f"Invalid line, AUTO_REMOVE_INVALID=False: {key}")
                    repair_log.append({"split": split_name, "file": label_path.name,
                                        "line_number": line_number,
                                        "action": "auto_remove_invalid",
                                        "reason": ";".join(parsed["reasons"])})
                    continue

                if parsed["canonical"] in seen:
                    if not AUTO_REMOVE_EXACT_DUPLICATES:
                        raise RuntimeError(f"Exact duplicate, AUTO_REMOVE_EXACT_DUPLICATES=False: {key}")
                    repair_log.append({"split": split_name, "file": label_path.name,
                                        "line_number": line_number,
                                        "action": "auto_remove_exact_duplicate",
                                        "reason": "same canonical annotation"})
                    continue
                seen.add(parsed["canonical"])
                output_lines.append(normalise_output_line(parsed))

            (out_labels / label_path.name).write_text("".join(output_lines), encoding="utf-8")

        orphan_labels = set(label_by_stem) - set(img_by_stem)
        for stem in sorted(orphan_labels):
            repair_log.append({"split": split_name, "file": f"{stem}.txt",
                                "line_number": "", "action": "drop_orphan_label",
                                "reason": "label_without_image"})
    return pd.DataFrame(repair_log)

repair_log_df = copy_clean_dataset(SOURCE_SPLITS, CLEAN_DIR, validation_df, MANUAL_DECISIONS)
repair_log_path = AUDIT_DIR / "repair_log.csv"
repair_log_df.to_csv(repair_log_path, index=False)
print(f"clean_v1 dibuat di: {CLEAN_DIR}")
print(f"Repair log        : {repair_log_path}")
display(repair_log_df["action"].value_counts().rename_axis("action").reset_index(name="count"))


## 8b. Class-Aware Copy-Paste Augmentation

Strategi: untuk setiap kelas minoritas, kumpulkan semua polygon mask yang valid
dari split `train` sebagai donor. Kemudian paste polygon tersebut ke gambar lain
dari split train secara acak.

Output disimpan langsung ke `clean_v1/train/images` dan `clean_v1/train/labels`
sehingga langsung masuk ke pipeline training tanpa langkah tambahan.

**Gambar sintetis tidak masuk ke `valid` atau `test`** — split tersebut tetap murni.


In [ ]:
def load_clean_annotations(clean_dir, split_name, class_names):
    """Baca semua anotasi valid dari clean_v1 split tertentu."""
    label_dir = clean_dir / split_name / "labels"
    image_dir = clean_dir / split_name / "images"
    records   = []
    for label_path in sorted(label_dir.glob("*.txt")):
        stem = label_path.stem
        img_candidates = [
            image_dir / f"{stem}{ext}"
            for ext in IMAGE_EXTENSIONS
            if (image_dir / f"{stem}{ext}").exists()
        ]
        if not img_candidates:
            continue
        image_path = img_candidates[0]
        lines = label_path.read_text(encoding="utf-8").splitlines()
        for line in lines:
            parsed = parse_annotation_line(line, 0, class_names)
            if not parsed["valid"]:
                continue
            records.append({
                "stem":       stem,
                "image_path": image_path,
                "label_path": label_path,
                "class_id":   parsed["class_id"],
                "class_name": class_names[parsed["class_id"]],
                "coords":     parsed["coords"],
                "canonical":  parsed["canonical"],
            })
    return records


def paste_polygon_to_image(
    base_image: Image.Image,
    donor_image: Image.Image,
    donor_coords: list,
    jitter_range: float = 0.05,
    rng=None,
):
    """
    Potong region polygon dari donor_image, lalu paste ke base_image
    di posisi acak (dengan sedikit jitter dari posisi asli).

    Mengembalikan (new_image, new_coords) atau None jika gagal.
    """
    if rng is None:
        rng = random.Random()

    try:
        bw, bh = base_image.size
        dw, dh = donor_image.size

        points_norm = np.asarray(donor_coords).reshape(-1, 2)
        points_donor_px = (points_norm * np.array([dw, dh])).astype(int)

        # Bbox region donor
        x_min_d = max(0, points_donor_px[:, 0].min())
        y_min_d = max(0, points_donor_px[:, 1].min())
        x_max_d = min(dw, points_donor_px[:, 0].max())
        y_max_d = min(dh, points_donor_px[:, 1].max())

        if x_max_d <= x_min_d or y_max_d <= y_min_d:
            return None

        # Crop region dari donor
        crop = donor_image.crop((x_min_d, y_min_d, x_max_d, y_max_d))
        crop_w, crop_h = crop.size

        # Skala ke base image jika donor crop terlalu besar
        max_fraction = 0.45
        if crop_w > bw * max_fraction or crop_h > bh * max_fraction:
            scale = min(bw * max_fraction / crop_w, bh * max_fraction / crop_h)
            crop_w = max(1, int(crop_w * scale))
            crop_h = max(1, int(crop_h * scale))
            crop   = crop.resize((crop_w, crop_h), Image.BILINEAR)

        # Posisi paste: jitter acak di sekitar tengah base
        center_x = rng.uniform(crop_w / 2, bw - crop_w / 2)
        center_y = rng.uniform(crop_h / 2, bh - crop_h / 2)
        paste_x  = int(center_x - crop_w / 2)
        paste_y  = int(center_y - crop_h / 2)
        paste_x  = max(0, min(paste_x, bw - crop_w))
        paste_y  = max(0, min(paste_y, bh - crop_h))

        # Mask sederhana (bounding box) untuk paste
        new_image = base_image.copy()
        new_image.paste(crop, (paste_x, paste_y))

        # Hitung koordinat polygon baru dalam normalized space
        scale_x = crop_w / (x_max_d - x_min_d) if (x_max_d - x_min_d) > 0 else 1.0
        scale_y = crop_h / (y_max_d - y_min_d) if (y_max_d - y_min_d) > 0 else 1.0

        points_orig_local = (points_norm * np.array([dw, dh])) - np.array([x_min_d, y_min_d])
        points_new_px     = points_orig_local * np.array([scale_x, scale_y]) + np.array([paste_x, paste_y])
        points_new_norm   = points_new_px / np.array([bw, bh])
        points_new_norm   = np.clip(points_new_norm, 0.0, 1.0)

        new_coords = points_new_norm.reshape(-1).tolist()
        return new_image, new_coords

    except Exception as exc:
        print(f"    [skip paste] {exc}")
        return None


def run_copy_paste_augmentation(
    clean_dir,
    class_names,
    minority_classes,
    copies_per_class=COPY_PASTE_PER_CLASS,
    donor_min_points=DONOR_MIN_POINTS,
    seed=RANDOM_SEED,
):
    """
    Untuk setiap kelas minoritas: buat `copies_per_class` gambar sintetis
    dengan cara paste instance kelas tersebut ke gambar lain.
    Gambar sintetis disimpan di clean_v1/train/images dan clean_v1/train/labels.
    """
    rng = random.Random(seed)
    train_records = load_clean_annotations(clean_dir, "train", class_names)
    if not train_records:
        print("Tidak ada anotasi train yang valid — skip augmentasi.")
        return []

    out_image_dir = clean_dir / "train" / "images"
    out_label_dir = clean_dir / "train" / "labels"
    created_log   = []

    # Kumpulkan semua gambar train sebagai target paste
    all_train_images = sorted(out_image_dir.glob("*"))
    all_train_images = [p for p in all_train_images if p.suffix.lower() in IMAGE_EXTENSIONS]

    for cls_name in minority_classes:
        cls_id = CLASS_TO_ID.get(cls_name)
        if cls_id is None:
            print(f"  [{cls_name}] tidak ditemukan di CLASS_TO_ID — skip.")
            continue

        # Donor: semua instance kelas ini di train
        donors = [r for r in train_records
                  if r["class_id"] == cls_id and r["coords"] is not None
                  and len(r["coords"]) // 2 >= donor_min_points]
        if not donors:
            print(f"  [{cls_name}] tidak ada donor yang memenuhi syarat — skip.")
            continue

        # Background: gambar yang TIDAK mengandung kelas ini (agar lebih diverse)
        cls_stems = {r["stem"] for r in train_records if r["class_id"] == cls_id}
        bg_images = [p for p in all_train_images if p.stem not in cls_stems]
        if len(bg_images) < 3:
            # Fallback: pakai semua gambar train jika bg tidak cukup
            bg_images = all_train_images

        print(f"  [{cls_name}] donors={len(donors)}, backgrounds={len(bg_images)}")

        n_created = 0
        attempts  = 0
        max_attempts = copies_per_class * 5

        while n_created < copies_per_class and attempts < max_attempts:
            attempts += 1
            donor_rec = rng.choice(donors)
            bg_path   = rng.choice(bg_images)

            try:
                base_img  = Image.open(bg_path).convert("RGB")
                donor_img = Image.open(donor_rec["image_path"]).convert("RGB")
            except Exception:
                continue

            result = paste_polygon_to_image(
                base_img, donor_img, donor_rec["coords"], rng=rng
            )
            if result is None:
                continue

            new_image, new_coords = result

            # Nama file unik
            synth_stem   = f"synth_{cls_name}_{seed}_{n_created:04d}"
            synth_img_p  = out_image_dir / f"{synth_stem}.jpg"
            synth_lbl_p  = out_label_dir / f"{synth_stem}.txt"

            # Simpan gambar
            new_image.save(synth_img_p, "JPEG", quality=92)

            # Salin anotasi dari background (gambar asli), tambah baris donor
            bg_label_p   = out_label_dir / f"{bg_path.stem}.txt"
            existing_lines = []
            if bg_label_p.exists():
                existing_lines = [
                    ln for ln in bg_label_p.read_text(encoding="utf-8").splitlines()
                    if ln.strip()
                ]

            # Baris baru untuk instance yang di-paste
            new_line = canonical_line(cls_id, new_coords)
            synth_lbl_p.write_text(
                "\n".join(existing_lines + [new_line]) + "\n",
                encoding="utf-8"
            )

            created_log.append({
                "synth_stem": synth_stem,
                "class_name": cls_name,
                "donor_stem": donor_rec["stem"],
                "bg_stem":    bg_path.stem,
            })
            n_created += 1

        print(f"    → dibuat {n_created} gambar sintetis (dari {attempts} percobaan)")

    return created_log


print("Menjalankan Class-Aware Copy-Paste Augmentation...")
print(f"Target kelas: {MINORITY_CLASSES}")
print(f"Jumlah per kelas: {COPY_PASTE_PER_CLASS}")
print()

copy_paste_log = run_copy_paste_augmentation(
    CLEAN_DIR, CLASS_NAMES, MINORITY_CLASSES,
    copies_per_class=COPY_PASTE_PER_CLASS,
    donor_min_points=DONOR_MIN_POINTS,
    seed=RANDOM_SEED,
)

copy_paste_df = pd.DataFrame(copy_paste_log)
if not copy_paste_df.empty:
    copy_paste_log_path = AUDIT_DIR / "copy_paste_log.csv"
    copy_paste_df.to_csv(copy_paste_log_path, index=False)
    print(f"\nTotal gambar sintetis dibuat: {len(copy_paste_df)}")
    display(copy_paste_df["class_name"].value_counts().rename_axis("class").reset_index(name="synth_count"))
else:
    print("Tidak ada gambar sintetis yang dibuat.")


## 8c. Targeted Oversampling — train.txt manifest

Buat `train.txt` yang berisi path gambar train, di mana gambar yang mengandung
kelas minoritas diulang sebanyak `OVERSAMPLE_FACTOR` kali. Gambar yang tidak
mengandung kelas minoritas tetap muncul 1 kali.

File ini digunakan di `data.yaml` sebagai `train:` alih-alih `train/images`.
Dengan cara ini model melihat kelas langka lebih sering per epoch — tanpa
menyentuh dataset fisik sama sekali.

> **Catatan:** Ultralytics mendukung `train.txt` manifest sejak YOLOv8.
> Pastikan path di YAML menggunakan path ke file `.txt`, bukan folder.


In [ ]:
def build_oversampling_manifest(
    clean_dir,
    class_names,
    minority_classes,
    oversample_factor=OVERSAMPLE_FACTOR,
    seed=RANDOM_SEED,
):
    """
    Buat train.txt dengan repetisi path gambar untuk kelas minoritas.
    Mengembalikan path ke file manifest yang dihasilkan.
    """
    rng = random.Random(seed)

    train_image_dir = clean_dir / "train" / "images"
    train_label_dir = clean_dir / "train" / "labels"

    # Identifikasi gambar mana yang mengandung kelas minoritas
    minority_ids  = {CLASS_TO_ID[cls] for cls in minority_classes if cls in CLASS_TO_ID}
    all_images    = sorted(train_image_dir.glob("*"))
    all_images    = [p for p in all_images if p.suffix.lower() in IMAGE_EXTENSIONS]

    entries = []          # list of (path_str, is_minority)
    minority_count = 0
    regular_count  = 0

    for img_path in all_images:
        label_path = train_label_dir / f"{img_path.stem}.txt"
        is_minority = False
        if label_path.exists():
            for line in label_path.read_text(encoding="utf-8").splitlines():
                tokens = line.strip().split()
                if tokens and int(tokens[0]) in minority_ids:
                    is_minority = True
                    break
        entries.append((str(img_path), is_minority))
        if is_minority:
            minority_count += 1
        else:
            regular_count  += 1

    # Bangun manifest dengan repetisi
    manifest_lines = []
    for img_path_str, is_minority in entries:
        repeat = oversample_factor if is_minority else 1
        manifest_lines.extend([img_path_str] * repeat)

    # Acak urutan supaya epoch tidak selalu dalam urutan yang sama
    rng.shuffle(manifest_lines)

    manifest_path = clean_dir / "train_oversampled.txt"
    manifest_path.write_text("\n".join(manifest_lines) + "\n", encoding="utf-8")

    print(f"Manifest ditulis ke: {manifest_path}")
    print(f"  Total entri unik  : {len(all_images)}")
    print(f"  Gambar minority   : {minority_count} (×{oversample_factor} = {minority_count * oversample_factor})")
    print(f"  Gambar regular    : {regular_count}")
    print(f"  Total entri di manifest: {len(manifest_lines)}")

    return manifest_path


manifest_path = build_oversampling_manifest(
    CLEAN_DIR, CLASS_NAMES, MINORITY_CLASSES,
    oversample_factor=OVERSAMPLE_FACTOR,
    seed=RANDOM_SEED,
)


## 9. Re-audit clean_v1

In [ ]:
def make_split_dirs_from_root(root):
    return {
        split_name: {
            "name": split_name, "root": root / split_name,
            "images": root / split_name / "images",
            "labels": root / split_name / "labels",
        }
        for split_name in SPLIT_NAMES
    }

CLEAN_SPLITS     = make_split_dirs_from_root(CLEAN_DIR)
CLEAN_VALIDATION = collect_validation(CLEAN_SPLITS, CLASS_NAMES)
CLEAN_PAIRING    = pairing_table(CLEAN_SPLITS)

reaudit_summary = []
for split_name in SPLIT_NAMES:
    split_rows  = CLEAN_VALIDATION[CLEAN_VALIDATION["split"] == split_name]
    pairing_row = CLEAN_PAIRING[CLEAN_PAIRING["split"] == split_name].iloc[0]
    reaudit_summary.append({
        "split":                split_name,
        "images":               len(image_files(CLEAN_SPLITS[split_name]["images"])),
        "label_files":          len(label_files(CLEAN_SPLITS[split_name]["labels"])),
        "invalid_lines":        int((
            (~split_rows["valid"]) & split_rows["format"].ne("empty_file")
        ).sum()),
        "exact_duplicates":     int(split_rows["exact_duplicate"].sum()),
        "images_without_labels": int(pairing_row["images_without_labels"]),
        "labels_without_images": int(pairing_row["labels_without_images"]),
    })

reaudit_summary_df = pd.DataFrame(reaudit_summary)
display(reaudit_summary_df)

hard_clean = (
    int(reaudit_summary_df["invalid_lines"].sum()) == 0
    and int(reaudit_summary_df["exact_duplicates"].sum()) == 0
    and int(reaudit_summary_df["images_without_labels"].sum()) == 0
    and int(reaudit_summary_df["labels_without_images"].sum()) == 0
)

if not hard_clean:
    print("RE-AUDIT GAGAL: masih ada hard error. Jangan lanjut ke fine-tuning.")
else:
    print("RE-AUDIT LULUS untuk hard validation checks.")
    print("Geometry flags tetap merupakan kandidat review visual, bukan error format.")


## 10. Clean dataset export dan file laporan

In [ ]:
if not hard_clean:
    raise RuntimeError("Export final dihentikan karena re-audit masih menemukan hard error.")

# ── data.yaml standar (train folder) ────────────────────────────────────────
clean_yaml_standard = {
    "path":  ".",
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    NUM_CLASSES,
    "names": CLASS_NAMES,
}
with (CLEAN_DIR / "data.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(clean_yaml_standard, f, sort_keys=False, allow_unicode=True)

# ── data_oversampled.yaml (gunakan manifest train.txt) ───────────────────────
# Gunakan file ini saat training jika ingin manfaatkan targeted oversampling.
clean_yaml_oversampled = {
    "path":  str(CLEAN_DIR),
    "train": str(manifest_path),   # ← train.txt manifest
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    NUM_CLASSES,
    "names": CLASS_NAMES,
}
with (CLEAN_DIR / "data_oversampled.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(clean_yaml_oversampled, f, sort_keys=False, allow_unicode=True)

print("data.yaml            →", CLEAN_DIR / "data.yaml")
print("data_oversampled.yaml→", CLEAN_DIR / "data_oversampled.yaml")

# ── Audit reports ────────────────────────────────────────────────────────────
overview_df.to_csv(AUDIT_DIR / "source_overview.csv", index=False)
pairing_df.drop(columns=["unpaired_image_names","orphan_label_names"]).to_csv(
    AUDIT_DIR / "source_pairing.csv", index=False)
format_df.to_csv(AUDIT_DIR / "source_format_counts.csv", index=False)
error_df.to_csv(AUDIT_DIR / "source_validation_errors.csv", index=False)
class_distribution.to_csv(AUDIT_DIR / "source_class_distribution.csv", index=False)
geometry_df.to_csv(AUDIT_DIR / "source_geometry.csv", index=False)
class_geometry_stats.to_csv(AUDIT_DIR / "source_class_geometry.csv", index=False)
reaudit_summary_df.to_csv(AUDIT_DIR / "clean_reaudit_summary.csv", index=False)

# ── Zip ──────────────────────────────────────────────────────────────────────
zip_path = WORK_ROOT / "clean_v1.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in CLEAN_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(CLEAN_DIR.parent))

print("=" * 72)
print("CLEAN DATASET READY")
print("=" * 72)
print("YAML (standard)   :", CLEAN_DIR / "data.yaml")
print("YAML (oversampled):", CLEAN_DIR / "data_oversampled.yaml")
print("ZIP               :", zip_path)
print("Audit             :", AUDIT_DIR)

# ── Ringkasan kelas setelah augmentasi ───────────────────────────────────────
FINAL_SPLITS = make_split_dirs_from_root(CLEAN_DIR)
final_train_records = load_clean_annotations(CLEAN_DIR, "train", CLASS_NAMES)
final_class_counts = Counter(r["class_name"] for r in final_train_records)
print("\nDistribusi kelas train setelah augmentasi:")
for cls_name in sorted(final_class_counts, key=final_class_counts.get):
    original = sum(
        1 for r in train_records
        if r["class_name"] == cls_name
    ) if "train_records" in dir() else "?"
    print(f"  {cls_name:<15} {final_class_counts[cls_name]:>4} instances")


## 11. Download hasil dari Kaggle

In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(WORK_ROOT / "clean_v1.zip")))
display(FileLink(str(AUDIT_DIR / "repair_decisions_template.csv")))
display(FileLink(str(AUDIT_DIR / "source_geometry.csv")))
display(FileLink(str(AUDIT_DIR / "copy_paste_log.csv") if (AUDIT_DIR / "copy_paste_log.csv").exists() else str(AUDIT_DIR / "repair_log.csv")))


---
## Checklist sebelum notebook fine-tuning

- upload atau gunakan `clean_v1.zip` sebagai dataset Kaggle baru (`nutrivision-pro-yolo-cleanv1`)
- pastikan `data.yaml` menunjuk ke `train/images`, `valid/images`, dan `test/images`
- tinjau `repair_decisions_template.csv` untuk kasus geometry yang mencurigakan
- untuk training dengan oversampling: gunakan `data_oversampled.yaml` sebagai `DATA_YAML` di stage 1/2
- untuk training standar: gunakan `data.yaml`
- lanjutkan training hanya jika `hard_clean=True`
